# BA-13 – Prescriptive Business Rules

This notebook converts BA-12 risk rankings into human-supervised maintenance and operational recommendations. It does not issue autonomous control commands.


## Step 1 — Upload BA-12 dashboard dataset
Upload `BA-12_dashboard_risk_dataset.csv`.


In [ ]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
source_file = 'BA-12_dashboard_risk_dataset.csv'
if source_file not in uploaded:
    raise FileNotFoundError(f'Please upload {source_file}')
df = pd.read_csv(source_file)
print('Loaded:', df.shape)
display(df.head())


In [ ]:
rules = {
    'Critical': {
        'priority':'P1',
        'action':'Immediate technical review / inspection',
        'operational_response':'Escalate to responsible maintenance/operations lead; validate condition and arrange immediate inspection.',
        'planning_response':'Create or prioritize a maintenance work item after human validation; consider restricting non-essential use only when separately authorized.',
        'evidence':'Recent alarms/alerts, operating condition indicators, maintenance history, outstanding defects, recent abnormal events.'
    },
    'High': {
        'priority':'P2',
        'action':'Prioritize technical review and planned inspection',
        'operational_response':'Escalate for technical review and verify whether inspection or maintenance can be brought forward.',
        'planning_response':'Schedule inspection or planned maintenance within the approved maintenance planning window, subject to human review.',
        'evidence':'Recent trend/condition indicators, maintenance history, recurring defects, operating context.'
    },
    'Lower': {
        'priority':'P3',
        'action':'Continue monitoring; no automatic intervention',
        'operational_response':'Maintain routine operational monitoring and normal escalation processes.',
        'planning_response':'Keep routine inspection/maintenance cadence unless other evidence indicates a change in condition.',
        'evidence':'Routine condition indicators, recent maintenance history, newly reported defects or alarms.'
    }
}

def map_rule(row, key):
    return rules[row['Risk_Category']][key]

df['Prescriptive_Action'] = df.apply(lambda r: map_rule(r,'action'), axis=1)
df['Operational_Response'] = df.apply(lambda r: map_rule(r,'operational_response'), axis=1)
df['Planning_Response'] = df.apply(lambda r: map_rule(r,'planning_response'), axis=1)
df['Evidence_To_Check'] = df.apply(lambda r: map_rule(r,'evidence'), axis=1)
df['Action_Rationale'] = df['Risk_Category'].map({
    'Critical':'Highest-risk band requires immediate human-led validation and inspection.',
    'High':'Elevated risk warrants prioritized technical review and planned inspection.',
    'Lower':'Lower predicted risk supports continued monitoring without automatic intervention.'
})
df['Decision_Owner'] = 'Maintenance/Operations responsible person'
df['Human_Approval_Required'] = 'Yes'
df['Autonomous_Control'] = 'No'

output_cols = ['Risk_Rank','Observation_ID','Risk_Score','Risk_Category','Priority_Level','Recommended_Action','Review_Status','Prescriptive_Action','Operational_Response','Planning_Response','Evidence_To_Check','Action_Rationale','Decision_Owner','Human_Approval_Required','Autonomous_Control']
recommendations = df[output_cols].copy()
recommendations.to_csv('BA-13_prescriptive_recommendations.csv', index=False)
summary = recommendations.groupby(['Risk_Category','Priority_Level'], as_index=False).agg(Observation_Count=('Observation_ID','size'), Mean_Risk_Score=('Risk_Score','mean'), Max_Risk_Score=('Risk_Score','max'))
summary['Prescriptive_Action'] = summary['Risk_Category'].map({k:v['action'] for k,v in rules.items()})
summary.to_csv('BA-13_prescriptive_summary.csv', index=False)
display(summary)


## Step 2 — Validate outputs
Check that every recommendation has human approval required = Yes and autonomous control = No.


In [ ]:
assert recommendations['Human_Approval_Required'].eq('Yes').all()
assert recommendations['Autonomous_Control'].eq('No').all()
assert set(recommendations['Risk_Category'].unique()).issubset({'Critical','High','Lower'})
print('Validation passed.')
print('Recommendations:', len(recommendations))


## Step 3 — Download BA-13 deliverables


In [ ]:
from google.colab import files
for f in ['BA-13_prescriptive_recommendations.csv','BA-13_prescriptive_summary.csv']:
    files.download(f)
